# Mew v2: fine-tuning the in-browser gate

Mew decides *whether* the tutor should respond. v1 asks a Gemini model on the server (~0.8 s). v2 is a small text classifier running in the browser (target: under 30 ms), trained on decisions students labelled in **Teach Mew**. When v2 isn't sure enough, v1 still decides.

Steps: **1** build the dataset, **2** fine-tune and export, **3** check it, **4** install it in the web app.

Run this with the `ml/` environment (`cd ml && uv sync`, then pick its `.venv` as the kernel). The dataset contains students' words: keep it on a trusted machine and never commit it (`ml/data/` is gitignored).

## 1. The dataset

The exporter lives with the API service (it shares its Supabase code). Sources:
- `--supabase`: labelled decisions of students who are **still** opted in (needs `SUPABASE_SECRET_KEY` in `.env.local`);
- `--jsonl`: files downloaded from Teach Mew;
- `--seed`: the v1 gate's eval cases.

Each decision becomes one line of text (`mew-features.v1`, identical in Python and in the browser), with a deterministic 80/10/10 split.

In [ ]:
import json, subprocess, sys
from pathlib import Path

ML = Path.cwd()
REPO = ML.parent
SOURCES = ["--seed", "../frontend/evals/mew.cases.json"]  # add "--supabase" or "--jsonl", "labels.jsonl"
subprocess.run(
    ["uv", "run", "--frozen", "python", "-m", "mew_api.dataset", "--out", str(ML / "data"), *SOURCES],
    cwd=REPO / "backend",
    check=True,
)
json.loads((ML / "data" / "meta.json").read_text())

## 2. Fine-tune and export

`train.py` fine-tunes the base model, picks the **confidence threshold** on the validation split (the lowest at which answers are 95 % precise: below it the browser defers to v1), then exports ONNX quantized to int8 in the layout transformers.js expects.

Base models: `distilbert-base-uncased` (66 M parameters, ~65 MB quantized) is a good default; `answerdotai/ModernBERT-base` is stronger but larger. For a quick check of the pipeline, `prajjwal1/bert-tiny` (4 M) trains in seconds. A useful model needs a few hundred labelled decisions per label, not just the seed cases.

In [ ]:
BASE_MODEL = "distilbert-base-uncased"
subprocess.run(
    [sys.executable, "train.py", "--data", "data", "--model", BASE_MODEL, "--epochs", "8", "--out", "output/mew-gate"],
    cwd=ML,
    check=True,
)
report = json.loads((ML / "output" / "mew-gate" / "mew.json").read_text())
report

## 3. Check it

Run the quantized model (exactly what the browser will run) on the v1 gate's eval cases, and time it. Look at **coverage** (how often v2 is sure enough to answer) and **precision when sure** in the report above: a model that is rarely sure is safe, just not much faster.

In [ ]:
import time
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

out = ML / "output" / "mew-gate"
tokenizer = AutoTokenizer.from_pretrained(out)
session = ort.InferenceSession(str(out / "onnx" / "model_quantized.onnx"))
labels = report["labels"]
needed = {i.name for i in session.get_inputs()}

rows = [json.loads(l) for l in (ML / "data" / "test.jsonl").read_text().splitlines() if l]
right = sure = 0
started = time.perf_counter()
for row in rows:
    enc = tokenizer(row["text"], return_tensors="np", truncation=True, max_length=128)
    logits = session.run(["logits"], {k: v.astype(np.int64) for k, v in enc.items() if k in needed})[0][0]
    probs = np.exp(logits - logits.max()); probs /= probs.sum()
    best = int(probs.argmax())
    if probs[best] >= report["min_confidence"]:
        sure += 1
        right += labels[best] == row["label"]
per = (time.perf_counter() - started) / max(1, len(rows)) * 1000
print(f"test: {len(rows)} · sure {sure} · right when sure {right} · {per:.1f} ms each")

## 4. Install it in the web app

The browser loads the model from `frontend/public/workers/models/mew-gate/` (gitignored: deploy it with the app, like a build artifact). The next page load picks it up; the debug panel then shows `local` as the source of Mew's decisions. Delete the folder to go back to v1.

In [ ]:
import shutil

target = REPO / "frontend" / "public" / "workers" / "models" / "mew-gate"
if target.exists():
    shutil.rmtree(target)
(target / "onnx").mkdir(parents=True)
for name in ["config.json", "mew.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "vocab.txt"]:
    if (out / name).exists():
        shutil.copy(out / name, target / name)
shutil.copy(out / "onnx" / "model_quantized.onnx", target / "onnx")
sorted(p.name for p in target.rglob("*"))